# Baseline: MedGemma sem Fine-tuning
Testa o MedGemma 4B no ISIC antes de qualquer ajuste, para ter uma linha de base.

In [ ]:
# Instala dependências (apenas no Kaggle)
!pip install -q transformers accelerate huggingface_hub

In [ ]:
import sys
sys.path.append('/kaggle/working/melanoma-tcc')

import pandas as pd
from kaggle_secrets import UserSecretsClient
from src.data.preprocessing import ISICDataset, split_dataframe
from src.model.inference import load_model, predict, extract_label_from_response
from src.utils.metrics import compute_metrics, plot_confusion_matrix

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret('HF_TOKEN')

CSV_PATH = '/kaggle/input/competitions/siim-isic-melanoma-classification/train.csv'
IMAGES_DIR = '/kaggle/input/competitions/siim-isic-melanoma-classification/jpeg/train'

In [ ]:
model, processor = load_model(HF_TOKEN)

In [ ]:
# Amostra pequena para o baseline (evita custo de GPU)
df = pd.read_csv(CSV_PATH)
sample_df = pd.concat([
    df[df['target'] == 1].sample(50, random_state=42),
    df[df['target'] == 0].sample(50, random_state=42),
]).reset_index(drop=True)

sample_df.to_csv('/kaggle/working/sample.csv', index=False)
dataset = ISICDataset('/kaggle/working/sample.csv', IMAGES_DIR, processor)

In [ ]:
labels, predictions, responses = [], [], []

for i in range(len(dataset)):
    sample = dataset[i]
    response = predict(model, processor, sample['image'], sample['prompt'])
    pred_label = extract_label_from_response(response)
    labels.append(sample['label'])
    predictions.append(pred_label)
    responses.append(response)
    if i % 10 == 0:
        print(f'[{i}/{len(dataset)}] label={sample["label"]} pred={pred_label}')

print('Inferencia concluida.')

In [ ]:
valid = [(l, p) for l, p in zip(labels, predictions) if p != -1]
valid_labels, valid_preds = zip(*valid)
results = compute_metrics(list(valid_labels), list(valid_preds))
plot_confusion_matrix(list(valid_labels), list(valid_preds), save_path='/kaggle/working/baseline_cm.png')